In [12]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("DataFramePractice") \
    .master("yarn") \
    .config("spark.executor.memory", "2g") \
    .getOrCreate()

df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv("/user/root/image-pipeline/input/users.csv")

df.printSchema()
df.show(5)

root
 |-- id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- city: string (nullable = true)

+---+-------+---+--------+
| id|   name|age|    city|
+---+-------+---+--------+
|  1|  Alice| 25| Beijing|
|  2|    Bob| 30|Shanghai|
|  3|Charlie| 35|Hangzhou|
|  4|  Diana| 28| Beijing|
|  5|    Eve| 32|Shanghai|
+---+-------+---+--------+
only showing top 5 rows



In [13]:
print("Age > 30:")
df.filter(df.age>30).show()

Age > 30:
+---+-------+---+--------+
| id|   name|age|    city|
+---+-------+---+--------+
|  3|Charlie| 35|Hangzhou|
|  5|    Eve| 32|Shanghai|
|  8|  Henry| 33|Shanghai|
|  9|   Iris| 31|Hangzhou|
+---+-------+---+--------+



In [7]:
print("group by city:")
df.groupBy("city").agg({"age":"avg","id":"count"}).show()

group by city:
+--------+---------+------------------+
|    city|count(id)|          avg(age)|
+--------+---------+------------------+
| Beijing|        4|              26.5|
|Hangzhou|        3|31.666666666666668|
|Shanghai|        3|31.666666666666668|
+--------+---------+------------------+



In [14]:
# 按年龄降序排列
print("Sorted by age desc:")
df.orderBy(df.age.desc()).show()

Sorted by age desc:
+---+-------+---+--------+
| id|   name|age|    city|
+---+-------+---+--------+
|  3|Charlie| 35|Hangzhou|
|  8|  Henry| 33|Shanghai|
|  5|    Eve| 32|Shanghai|
|  9|   Iris| 31|Hangzhou|
|  2|    Bob| 30|Shanghai|
|  6|  Frank| 29|Hangzhou|
|  4|  Diana| 28| Beijing|
|  7|  Grace| 27| Beijing|
| 10|   Jack| 26| Beijing|
|  1|  Alice| 25| Beijing|
+---+-------+---+--------+



In [10]:
# 保存为 Parquet 到 HDFS 输出目录
df.write.mode("overwrite").parquet("/user/root/image-pipeline/output/users_parquet")

# 读取验证
parquet_df = spark.read.parquet("/user/root/image-pipeline/output/users_parquet")
parquet_df.show()

+---+-------+---+--------+
| id|   name|age|    city|
+---+-------+---+--------+
|  1|  Alice| 25| Beijing|
|  2|    Bob| 30|Shanghai|
|  3|Charlie| 35|Hangzhou|
|  4|  Diana| 28| Beijing|
|  5|    Eve| 32|Shanghai|
|  6|  Frank| 29|Hangzhou|
|  7|  Grace| 27| Beijing|
|  8|  Henry| 33|Shanghai|
|  9|   Iris| 31|Hangzhou|
| 10|   Jack| 26| Beijing|
+---+-------+---+--------+



In [21]:
print("Spark UI:", spark.sparkContext.uiWebUrl)

Spark UI: http://master:4040


In [19]:
# 最直接的方式：查看 SparkContext 的环境信息
print("Spark Master:", sc.master)
print("应用 ID:", sc.applicationId)

# 通过 SparkConf 确认是否使用了 YARN
conf = sc.getConf()
print("部署模式:", conf.get("spark.master"))

Spark Master: yarn
应用 ID: application_1779795266245_0002
部署模式: yarn


In [22]:
sc = spark.sparkContext
status_tracker = sc._jsc.sc().statusTracker()

print("=== 活跃 Executor 信息 ===")
for info in status_tracker.getExecutorInfos():
    # 直接打印对象，会显示主机和端口等关键信息
    print(info.toString())

=== 活跃 Executor 信息 ===
org.apache.spark.SparkExecutorInfoImpl@654941cc
org.apache.spark.SparkExecutorInfoImpl@7e368932
org.apache.spark.SparkExecutorInfoImpl@1de4066b
